In [ ]:
# Created by D. S. Murray, June 2025

In [ ]:
import pandas as pd
from pandas.tseries.offsets import DateOffset
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import griddata
import datetime as dt
from pathlib import Path
import os
from tqdm import tqdm
from fnmatch import fnmatch


In [ ]:
#Define the function that opens the individual netcdf files, and selects the variables of interest w/ dimensions preserved 
# (this function helps speed up code processing time)

def process_daily_files(result_dir, files_to_open, variables_to_select, progress_bar=None):
    # Open the daily files as an xarray dataset
    ds = xr.open_mfdataset(str(result_dir / files_to_open), combine='nested')

    # Select specific variables
    ds = ds[variables_to_select]

    # Update progress bar
    if progress_bar:
        progress_bar.update(1)
    return ds

In [ ]:
#Create timeseries from individual netcdf files (in this case, daily files)
result_dir = Path('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/atm_h2')

files_to_open = 'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam.h2.*.nc'

variables_to_select = ['WD_NH3', 'so4_a1SFWET', 'so4_a2SFWET', 'so4_a3SFWET', 'so4_c1SFWET', 'so4_c2SFWET', 'so4_c3SFWET', 'WD_HNO3', 'WD_NH4']

files_list = list(result_dir.glob(files_to_open))

# Initialize tqdm progress bar
progress = tqdm(total=len(files_list))

processed_data = []
for file in files_list:
    processed_data.append(process_daily_files(result_dir, file.relative_to(result_dir), variables_to_select, progress_bar=progress))

progress.close()

nc_daily = xr.concat(processed_data, dim='time').sortby('time')
nc_daily

In [ ]:
#Write out an individual .nc file for each data variable (dimensions preserved)
for i in nc_daily.data_vars:
    print(i)
    dat = nc_daily[[i]]
    dat.to_netcdf(f'/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Timeseries_atm_h2/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.{i}.nc')